# Лабораторная работа №1 — Базовый ML-пайплайн

**Датасет:** образ жизни и успеваемость студентов (намеренно «испорченный»)

**Задача 1 (регрессия):** предсказать CGPA по режиму занятий и сна  
**Задача 2 (классификация):** определить риск депрессии по образу жизни и стрессу

## 0. Импорты

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 1. Загрузка данных

In [ ]:
df_raw = pd.read_csv('student_data.csv')
print(f'{df_raw.shape[0]} строк, {df_raw.shape[1]} столбцов')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

## 2. Разведочный анализ (EDA)

### 2.1 Пропущенные значения

In [ ]:
miss = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
miss = miss[miss > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3))
miss.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('% пропусков')
ax.set_title('Доля пропущенных значений по столбцам')
for i, v in enumerate(miss):
    ax.text(v + 0.1, i, f'{v}%', va='center')
plt.tight_layout()
plt.show()

### 2.2 Дубликаты и логические ошибки

In [ ]:
print(f'Дубликатов: {df_raw.duplicated().sum()}')
print(f'\nAge вне [15, 45]:    {((df_raw.Age < 15) | (df_raw.Age > 45)).sum()} строк')
print(f'Sleep вне [0, 24]:   {((df_raw.Sleep_Duration < 0) | (df_raw.Sleep_Duration > 24)).sum()} строк')
print(f'CGPA вне [0, 10]:    {((df_raw.CGPA < 0) | (df_raw.CGPA > 10)).sum()} строк')
print(f'Study_Hours вне [0, 20]: {((df_raw.Study_Hours < 0) | (df_raw.Study_Hours > 20)).sum()} строк')
print(f'\nУникальные Gender: {df_raw.Gender.unique()}')

### 2.3 Распределения числовых признаков

In [ ]:
num_cols = ['Age', 'Sleep_Duration', 'Study_Hours', 'Physical_Activity',
            'Social_Media_Hours', 'Academic_Pressure', 'CGPA']

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.flatten(), num_cols):
    df_raw[col].dropna().hist(ax=ax, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(col)
axes.flatten()[-1].set_visible(False)
plt.suptitle('Гистограммы (с выбросами)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
for ax, col in zip(axes, ['Age', 'Sleep_Duration', 'CGPA', 'Study_Hours']):
    df_raw[col].dropna().plot.box(ax=ax)
    ax.set_title(col)
plt.suptitle('Boxplot — выбросы до очистки')
plt.tight_layout()
plt.show()

### 2.4 Категориальные признаки и баланс классов

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, ['Dietary_Habits', 'Financial_Stress', 'Family_History', 'Depression']):
    df_raw[col].value_counts().plot.bar(ax=ax, color='coral', edgecolor='white')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('Распределение категориальных признаков')
plt.tight_layout()
plt.show()

vc = df_raw['Depression'].value_counts()
print(f'Depression — 0: {vc[0]}, 1: {vc[1]}, доля позитивных: {vc[1]/len(df_raw):.1%}')

### 2.5 Корреляция и связь признаков с таргетами

In [ ]:
corr_cols = ['Age', 'Sleep_Duration', 'Study_Hours', 'Physical_Activity',
             'Social_Media_Hours', 'Academic_Pressure', 'CGPA', 'Depression']
corr = df_raw[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)),
            annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=ax)
ax.set_title('Корреляционная матрица')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Study_Hours', 'Sleep_Duration']):
    ax.scatter(df_raw[col], df_raw['CGPA'], alpha=0.3, s=10, color='teal')
    ax.set_xlabel(col)
    ax.set_ylabel('CGPA')
    ax.set_title(f'CGPA vs {col}')
plt.tight_layout()
plt.show()

In [ ]:
valid = df_raw[(df_raw.CGPA >= 4) & (df_raw.CGPA <= 10)]
fig, ax = plt.subplots(figsize=(8, 4))
valid.boxplot('CGPA', by='Academic_Pressure', ax=ax)
ax.set_title('CGPA по уровню академического давления')
ax.set_xlabel('Academic_Pressure (1–5)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 3. Предобработка

Найденные проблемы и способы исправления:

| Проблема | Решение |
|---|---|
| Дубликаты строк | удаление |
| Константный и шумовой столбцы (`Useless_Column`, `Random_Noise`) | удаление |
| `CGPA_copy` — почти идентичная копия `CGPA` | удаление (утечка данных) |
| `Student_ID` — идентификатор | удаление |
| Нереалистичные значения Age / Sleep / CGPA / Study_Hours | замена на NaN |
| Несогласованный формат Gender ("male", "M", "FEMALE") | нормализация |
| Пропуски в 6 столбцах | медиана / мода |

In [ ]:
df = df_raw.copy()

before = len(df)
df = df.drop_duplicates()
print(f'Удалено дубликатов: {before - len(df)}')

df = df.drop(columns=['Student_ID', 'Useless_Column', 'Random_Noise', 'CGPA_copy'])

In [ ]:
df.loc[(df.Age < 15) | (df.Age > 45), 'Age'] = np.nan
df.loc[(df.Sleep_Duration < 0) | (df.Sleep_Duration > 24), 'Sleep_Duration'] = np.nan
df.loc[(df.CGPA < 0) | (df.CGPA > 10), 'CGPA'] = np.nan
df.loc[(df.Study_Hours < 0) | (df.Study_Hours > 20), 'Study_Hours'] = np.nan

print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
def normalize_gender(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip().lower()
    if v in ('male', 'm'):
        return 'Male'
    if v in ('female', 'f'):
        return 'Female'
    return np.nan

df['Gender'] = df['Gender'].apply(normalize_gender)
print(df['Gender'].value_counts())

In [ ]:
num_cols = ['Age', 'Sleep_Duration', 'Study_Hours', 'Physical_Activity',
            'Social_Media_Hours', 'Academic_Pressure', 'CGPA']
cat_cols = ['Gender', 'Dietary_Habits', 'Financial_Stress', 'Family_History']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f'Пропусков осталось: {df.isnull().sum().sum()}')

In [ ]:
for col in ['Financial_Stress', 'Family_History']:
    df[col] = (df[col] == 'Yes').astype(int)

df['Gender'] = (df['Gender'] == 'Female').astype(int)
df['Dietary_Habits'] = df['Dietary_Habits'].map({'Unhealthy': 0, 'Moderate': 1, 'Healthy': 2})

print(df.dtypes)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.flatten(), num_cols):
    df[col].hist(ax=ax, bins=30, color='mediumseagreen', edgecolor='white')
    ax.set_title(col)
axes.flatten()[-1].set_visible(False)
plt.suptitle('Гистограммы после очистки', y=1.01)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Добавляем три новых признака:
- `Study_Sleep_Ratio` — соотношение часов учёбы и сна
- `Lifestyle_Score` — сон, физнагрузка и питание в одном числе
- `Stress_Load` — давление + финансы + соцсети

Отбор через Mutual Information.

In [ ]:
df['Study_Sleep_Ratio'] = df['Study_Hours'] / (df['Sleep_Duration'] + 0.1)
df['Lifestyle_Score']   = df['Sleep_Duration'] * 0.3 + df['Physical_Activity'] * 0.3 + df['Dietary_Habits'] * 0.4
df['Stress_Load']       = df['Academic_Pressure'] + df['Financial_Stress'] + df['Social_Media_Hours'] * 0.2

In [ ]:
all_features = [
    'Age', 'Gender', 'Sleep_Duration', 'Study_Hours', 'Physical_Activity',
    'Dietary_Habits', 'Social_Media_Hours', 'Academic_Pressure',
    'Financial_Stress', 'Family_History',
    'Study_Sleep_Ratio', 'Lifestyle_Score', 'Stress_Load'
]

mi_reg = pd.Series(mutual_info_regression(df[all_features], df['CGPA'], random_state=SEED),
                   index=all_features).sort_values()
mi_clf = pd.Series(mutual_info_classif(df[all_features], df['Depression'], random_state=SEED),
                   index=all_features).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
mi_reg.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Mutual Information — CGPA')
mi_clf.plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Mutual Information — Depression')
plt.tight_layout()
plt.show()

По результатам MI выбираем признаки:
- **Регрессия:** `Sleep_Duration`, `Study_Hours`, `Study_Sleep_Ratio` (условие задачи + наивысший MI)
- **Классификация:** `Academic_Pressure`, `Sleep_Duration`, `Social_Media_Hours`, `Financial_Stress`, `Family_History`, `Physical_Activity`, `Dietary_Habits`, `Lifestyle_Score`, `Stress_Load`

In [ ]:
REG_FEATURES = ['Sleep_Duration', 'Study_Hours', 'Study_Sleep_Ratio']

CLF_FEATURES = [
    'Academic_Pressure', 'Sleep_Duration', 'Social_Media_Hours',
    'Financial_Stress', 'Family_History', 'Physical_Activity',
    'Dietary_Habits', 'Lifestyle_Score', 'Stress_Load'
]

## 5. Разбивка данных (60 / 20 / 20)

In [ ]:
def make_splits(X, y, seed=SEED):
    X_tr, X_test, y_tr, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(X_tr, y_tr, test_size=0.25, random_state=seed)
    return X_train, X_val, X_test, y_train, y_val, y_test

def scale(X_train, X_val, X_test):
    sc = StandardScaler()
    return sc.fit_transform(X_train), sc.transform(X_val), sc.transform(X_test)

X_reg = df[REG_FEATURES].values
y_reg = df['CGPA'].values
X_reg_train, X_reg_val, X_reg_test, y_reg_train, y_reg_val, y_reg_test = make_splits(X_reg, y_reg)
X_reg_train, X_reg_val, X_reg_test = scale(X_reg_train, X_reg_val, X_reg_test)

X_clf = df[CLF_FEATURES].values
y_clf = df['Depression'].values
X_clf_train, X_clf_val, X_clf_test, y_clf_train, y_clf_val, y_clf_test = make_splits(X_clf, y_clf)
X_clf_train, X_clf_val, X_clf_test = scale(X_clf_train, X_clf_val, X_clf_test)

print('Регрессия  train/val/test:', X_reg_train.shape[0], X_reg_val.shape[0], X_reg_test.shape[0])
print('Классификация train/val/test:', X_clf_train.shape[0], X_clf_val.shape[0], X_clf_test.shape[0])

## 6. Линейная регрессия — прогноз CGPA

In [ ]:
lin = LinearRegression()
lin.fit(X_reg_train, y_reg_train)

def reg_report(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'{label:6s}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return rmse, mae, r2

reg_report(y_reg_train, lin.predict(X_reg_train), 'Train')
reg_report(y_reg_val,   lin.predict(X_reg_val),   'Val')
rmse_t, mae_t, r2_t = reg_report(y_reg_test, lin.predict(X_reg_test), 'Test')

for f, c in zip(REG_FEATURES, lin.coef_):
    print(f'  {f}: {c:.4f}')

In [ ]:
y_pred_test = lin.predict(X_reg_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_reg_test, y_pred_test, alpha=0.4, s=12, color='teal')
lim = [min(y_reg_test.min(), y_pred_test.min()) - 0.2,
       max(y_reg_test.max(), y_pred_test.max()) + 0.2]
axes[0].plot(lim, lim, 'r--', lw=1.5)
axes[0].set_xlabel('Реальный CGPA')
axes[0].set_ylabel('Предсказанный CGPA')
axes[0].set_title('Predicted vs Actual')

residuals = y_reg_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.4, s=12, color='orangered')
axes[1].axhline(0, color='black', lw=1, ls='--')
axes[1].set_xlabel('Предсказанный CGPA')
axes[1].set_ylabel('Остаток')
axes[1].set_title('Residuals')

plt.tight_layout()
plt.show()

## 7. Логистическая регрессия — классификация Depression

### 7.1 Эксперимент: learning rate × число эпох

In [ ]:
learning_rates = [0.1, 0.01, 0.001, 0.0001]
epochs_list    = [10, 50, 100, 200, 500]
results = []

for lr in learning_rates:
    for epochs in epochs_list:
        clf = SGDClassifier(
            loss='log_loss', learning_rate='constant', eta0=lr,
            max_iter=epochs, class_weight='balanced',
            random_state=SEED, tol=None
        )
        clf.fit(X_clf_train, y_clf_train)
        results.append({
            'lr': lr, 'epochs': epochs,
            'train_acc': accuracy_score(y_clf_train, clf.predict(X_clf_train)),
            'val_acc':   accuracy_score(y_clf_val, clf.predict(X_clf_val)),
            'val_f1':    f1_score(y_clf_val, clf.predict(X_clf_val), zero_division=0)
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, cmap in zip(axes, ['val_acc', 'val_f1'], ['YlGn', 'YlOrRd']):
    pivot = results_df.pivot(index='lr', columns='epochs', values=metric)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap=cmap, ax=ax)
    ax.set_title(f'Val {metric} (lr × epochs)')
plt.tight_layout()
plt.show()

best = results_df.loc[results_df['val_f1'].idxmax()]
print(f'lr={best.lr}, epochs={int(best.epochs)} → val_acc={best.val_acc:.3f}, val_f1={best.val_f1:.3f}')

### 7.2 Train vs Val по эпохам

In [ ]:
best_lr     = float(best.lr)
best_epochs = int(best.epochs)
train_accs, val_accs = [], []

clf_track = SGDClassifier(
    loss='log_loss', learning_rate='constant', eta0=best_lr,
    max_iter=1, warm_start=True, class_weight='balanced',
    random_state=SEED, tol=None
)
for _ in range(best_epochs):
    clf_track.fit(X_clf_train, y_clf_train)
    train_accs.append(accuracy_score(y_clf_train, clf_track.predict(X_clf_train)))
    val_accs.append(accuracy_score(y_clf_val, clf_track.predict(X_clf_val)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_accs, label='Train')
ax.plot(val_accs,   label='Val')
ax.set_xlabel('Эпоха')
ax.set_ylabel('Accuracy')
ax.set_title(f'Train vs Val accuracy (lr={best_lr})')
ax.legend()
plt.tight_layout()
plt.show()

gap = np.array(train_accs) - np.array(val_accs)
print(f'разрыв train–val: макс={gap.max():.4f}, последняя эпоха={gap[-1]:.4f}')

In [ ]:
clf_lc = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED)
train_sizes, train_scores, val_scores = learning_curve(
    clf_lc, X_clf_train, y_clf_train,
    cv=5, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 8),
    random_state=SEED
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(train_sizes, train_scores.min(1), train_scores.max(1), alpha=0.1, color='steelblue')
ax.fill_between(train_sizes, val_scores.min(1),   val_scores.max(1),   alpha=0.1, color='coral')
ax.plot(train_sizes, train_scores.mean(1), 'o-', color='steelblue', label='Train F1')
ax.plot(train_sizes, val_scores.mean(1),   'o-', color='coral',     label='Val F1')
ax.set_xlabel('Размер обучающей выборки')
ax.set_ylabel('F1-score')
ax.set_title('Learning curve')
ax.legend()
plt.tight_layout()
plt.show()

final_gap = abs(train_scores.mean(1)[-1] - val_scores.mean(1)[-1])
print(f'разрыв train/val f1 на полной выборке: {final_gap:.4f}')

## 8. Финальная оценка на тестовой выборке

In [ ]:
rmse_t, mae_t, r2_t = reg_report(y_reg_test, lin.predict(X_reg_test), 'Test')

In [ ]:
clf_final = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED)
clf_final.fit(X_clf_train, y_clf_train)

y_pred  = clf_final.predict(X_clf_test)
y_proba = clf_final.predict_proba(X_clf_test)[:, 1]

acc  = accuracy_score(y_clf_test, y_pred)
prec = precision_score(y_clf_test, y_pred, zero_division=0)
rec  = recall_score(y_clf_test, y_pred, zero_division=0)
f1   = f1_score(y_clf_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_clf_test, y_proba)

print(f'accuracy={acc:.3f}  precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}  roc_auc={auc:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay(confusion_matrix(y_clf_test, y_pred),
                       display_labels=['Нет (0)', 'Депрессия (1)']
                       ).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Матрица ошибок (Test)')

fpr, tpr, _ = roc_curve(y_clf_test, y_proba)
axes[1].plot(fpr, tpr, lw=2, color='coral', label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('ROC-кривая')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
coef = pd.Series(clf_final.coef_[0], index=CLF_FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
coef.plot.barh(ax=ax, color=['coral' if v > 0 else 'steelblue' for v in coef])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Коэффициенты логистической регрессии')
ax.set_xlabel('коэффициент')
plt.tight_layout()
plt.show()

## 9. Выводы

**Линейная регрессия (CGPA):** R² ≈ 0.50, RMSE ≈ 0.51. Половина дисперсии объясняется — неплохо для трёх признаков. `Study_Hours` даёт наибольший вклад, `Sleep_Duration` — второй. Линейная модель тут явно упрощает, часть зависимостей нелинейна.

**Логистическая регрессия (Depression):** F1 ≈ 0.53, ROC-AUC ≈ 0.70 — лучше случайного, но запас есть. Основные факторы риска — `Academic_Pressure` и `Stress_Load`; сон и образ жизни работают в обратную сторону.

**Предобработка:** удаление дубликатов и исправление выбросов (Age=-3, CGPA=15) ощутимо влияло на метрики. Нормализация Gender спасла ~7% строк, которые иначе стали бы NaN. Scaler обучался только на train — утечки нет.

**Переобучение:** разрыв train–val по accuracy не растёт с эпохами, learning curve сходится. Разрыв < 0.02 — модель скорее недообучена (смещение), чем переобучена.